# DataMate — agent code-gen trên **public** (D20)

Sinh submission public bằng agent sinh code (`src/datamate/agent/`). Chạy theo **shard** (nhóm shop) để mỗi lần chạy < 12 giờ của Kaggle: `NSHARDS` lần chạy, mỗi lần đổi `SHARD = 0, 1, 2...` (có thể chạy song song trên tài khoản Kaggle của các thành viên).

**Yêu cầu** (xem `docs/KAGGLE.md`):
- Accelerator **GPU T4 x2**, **Internet: On**, notebook **Private**.
- Dataset Kaggle **private** chứa thư mục `public_test` (`questions.jsonl`, `rulebook.md`, `calendar.csv`, `data/`).
- Nên chạy bằng **Save Version → Save & Run All (Commit)**: chạy nền, đóng trình duyệt được, output giữ lại sau khi xong.
- **Resume**: nếu một lần chạy bị dừng giữa chừng, Add Input → output (zip) của lần đó; notebook tự nạp `replay.jsonl` cũ nên các shop đã chạy không tốn GPU nữa.

**Đầu ra**: `/kaggle/working/datamate_public_<model>_<TAG>_shard<i>of<n>.zip` (`replay.jsonl`, `shard<i>/submission.jsonl`, `shard<i>/log.jsonl`, log chạy). Tải về, **không commit**, **không mở log để đọc câu hỏi** (CLAUDE.md luật 1): lead chỉ chạy `eval/resubmit_agent.py` để gộp shard và tạo submission.

In [ ]:
# ---- cấu hình ----
REPO = "https://github.com/dinhkhoi124/DateMate.git"
BRANCH = "main"
MODEL_REPO = "Qwen/Qwen3-14B-AWQ"      # ứng viên khác: Qwen/Qwen3-8B-AWQ, Qwen/Qwen2.5-Coder-14B-Instruct-AWQ
NSHARDS = 3                            # 1000 câu / 14 shop -> ~330 câu/shard (~6-8 giờ trên T4 x2)
SHARD = 0                              # 0..NSHARDS-1: đổi cho từng lần chạy
SAMPLES = 5                            # k mẫu / câu
TAG = "pub1"                             # gắn vào tên zip để phân biệt các lần chạy
THINKING = False                       # True: Qwen3 thinking mode (chậm hơn nhiều, cần MAX_TOKENS lớn)
MAX_TOKENS = 1536                      # token sinh / mẫu; THINKING=True thì đặt 6144
MAX_LEN = 20480                        # prompt ~10-12k token
VLLM_VERSION = "0.10.1.1"              # bản pin của image (D17); không chạy được trên T4 thì dùng fallback
VLLM_FALLBACK = None                   # None: không thử bản khác (bản khác = replay lệch image)
TRANSFORMERS_VERSION = "4.55.4"        # vLLM 0.10.1.1 gọi tokenizer.all_special_tokens_extended, transformers 5.x đã bỏ
PY_VERSION = "3.12"                    # vLLM <= 0.10.1.1 cần Python < 3.13; kernel Kaggle đã là 3.13
RUN_SMOKE = True                       # 2 câu của shop đầu shard trước (thêm 1 lần load model; 2 câu này được replay lại)
ZIP_EVERY_MIN = 20                     # đóng gói kết quả dở dang định kỳ (session chết vẫn còn zip gần nhất)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

In [ ]:
# Kernel Kaggle là Python 3.13, vLLM 0.10.1.1 chỉ có wheel cho <= 3.12.
# -> venv riêng Python 3.12 bằng uv. Mọi lệnh pipeline chạy bằng PY (không phải kernel).
import subprocess
def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-3000:])
    return r.returncode

VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"
sh("pip install -q uv")
sh(f"uv venv -q --python {PY_VERSION} {VENV}")     # uv tự tải Python 3.12 nếu máy chưa có

def vllm_ok():
    code = ("import torch, vllm; assert torch.cuda.is_available(); "
            "x = torch.ones(2, 2, device='cuda'); assert float((x @ x).sum()) == 8.0; "
            "print('python', __import__('sys').version.split()[0], 'torch', torch.__version__, "
            "'cuda', torch.version.cuda, 'vllm', vllm.__version__, 'transformers', __import__('transformers').__version__, "
            "'cap', torch.cuda.get_device_capability()); "
            "from transformers import AutoTokenizer as T; assert hasattr(T, 'from_pretrained')")
    return sh(f'{PY} -c "{code}"') == 0

sh(f"uv pip install -q --python {PY} vllm=={VLLM_VERSION} transformers=={TRANSFORMERS_VERSION}")
ok = vllm_ok()
if not ok and VLLM_FALLBACK:
    print(f"vLLM {VLLM_VERSION} không chạy được -> thử {VLLM_FALLBACK}")
    sh(f"uv pip install -q --python {PY} vllm=={VLLM_FALLBACK} transformers=={TRANSFORMERS_VERSION}")
    ok = vllm_ok()
assert ok, "vLLM không chạy được: gửi output cell này cho lead"

In [ ]:
import os
import glob
!rm -rf /kaggle/working/repo && git clone -q --depth 1 -b {BRANCH} {REPO} /kaggle/working/repo
os.chdir("/kaggle/working/repo")
!git log --oneline -1
# code trên GitHub phải khớp notebook này (sửa trên máy lead mà chưa push thì Kaggle vẫn chạy bản cũ)
STALE = "coalesce_suffixes" not in open("src/datamate/agent/sandbox.py", encoding="utf8").read()
assert not STALE, f"nhánh {BRANCH} trên GitHub là bản cũ: nhắc lead push rồi chạy lại cell này"
# deps runtime giống image (docker/requirements.txt), giữ nguyên torch/vllm vừa cài
!uv pip install -q --python {PY} -r docker/requirements.txt transformers=={TRANSFORMERS_VERSION} 2>&1 | tail -3
!{PY} -c "import pandas, numpy, transformers; print('pandas', pandas.__version__, 'numpy', numpy.__version__, 'transformers', transformers.__version__)"

In [ ]:
# dataset public -> PUB (đọc thẳng từ /kaggle/input, read-only). Public không có answers.jsonl
hits = [p for p in glob.glob("/kaggle/input/**/questions.jsonl", recursive=True)
        if not os.path.exists(os.path.join(os.path.dirname(p), "answers.jsonl"))
        and os.path.exists(os.path.join(os.path.dirname(p), "rulebook.md"))]
assert hits, "không thấy thư mục public_test (questions.jsonl + rulebook.md, không có answers.jsonl) trong /kaggle/input"
PUB = os.path.dirname(sorted(hits)[0])
print("public:", PUB)
!ls {PUB} | head

In [ ]:
from huggingface_hub import snapshot_download
MODEL_NAME = MODEL_REPO.split("/")[-1]
MODEL_DIR = f"/tmp/models/{MODEL_NAME}"          # tên thư mục = model_id trong khoá replay
snapshot_download(MODEL_REPO, local_dir=MODEL_DIR)
!du -sh {MODEL_DIR}

In [ ]:
import torch
NGPU = torch.cuda.device_count()
OUT = f"/kaggle/working/out_{MODEL_NAME}_{TAG}"
os.makedirs(OUT, exist_ok=True)
os.environ.update({
    "PYTHONPATH": "src", "PYTHONHASHSEED": "0",
    "DATAMATE_LLM_DTYPE": "float16",            # T4 không có bf16; AWQ chạy fp16
    "DATAMATE_LLM_TP": str(min(NGPU, 2)),
    "DATAMATE_LLM_MAX_LEN": str(MAX_LEN),
    "DATAMATE_LLM_GPU_UTIL": "0.90",
    # khoá replay gồm model_id: thinking on/off phải khác khoá
    "DATAMATE_LLM_MODEL_ID": MODEL_NAME + ("-think" if THINKING else ""),
    "DATAMATE_LLM_THINKING": "1" if THINKING else "0",
    "DATAMATE_AGENT_SAMPLES": str(SAMPLES),
    "DATAMATE_AGENT_MAX_TOKENS": str(MAX_TOKENS),
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
})
if torch.cuda.get_device_capability()[0] < 8:   # T4 = 7.5: engine V1 của 0.10.1.1 không hỗ trợ, dùng V0
    os.environ["VLLM_USE_V1"] = "0"
    os.environ.setdefault("VLLM_ATTENTION_BACKEND", "XFORMERS")
    os.environ["DATAMATE_LLM_PREFIX_CACHE"] = "0"   # kernel Triton prefix-prefill không biên dịch được cho sm75
print("GPUs:", NGPU, {k: v for k, v in os.environ.items() if k.startswith("DATAMATE_")})
RUN = f"{PY} -m datamate.run --agent codegen --llm vllm --llm-model {MODEL_DIR} --llm-replay {OUT}/replay.jsonl"

# resume: replay của các lần chạy trước (Add Input -> output zip cũ). Khoá = prompt + model + sampling: câu đã có
# không gọi GPU nữa, câu mới được ghi tiếp vào cùng file
old = sorted(glob.glob("/kaggle/input/**/replay.jsonl", recursive=True))
if old and not os.path.exists(f"{OUT}/replay.jsonl"):
    with open(f"{OUT}/replay.jsonl", "w", encoding="utf8") as w:
        for p in old:
            w.write(open(p, encoding="utf8").read())
    print("resume từ", old)
!wc -l {OUT}/replay.jsonl 2>/dev/null || echo "replay mới"

In [ ]:
# chia shop vào NSHARDS nhóm cân theo số câu (deterministic: tham lam, tie-break theo mã shop)
# chỉ đếm business_id, không in câu hỏi
import json
from collections import Counter
cnt = Counter(json.loads(l)["business_id"] for l in open(f"{PUB}/questions.jsonl", encoding="utf8") if l.strip())
groups = [[] for _ in range(NSHARDS)]
for biz, n in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0])):
    i = min(range(NSHARDS), key=lambda j: (sum(cnt[b] for b in groups[j]), j))
    groups[i].append(biz)
for i, g in enumerate(groups):
    print(f"shard {i}: {sum(cnt[b] for b in g):4} câu  {sorted(g)}")
assert 0 <= SHARD < NSHARDS
SHOPS = sorted(groups[SHARD])
NQ = sum(cnt[b] for b in SHOPS)
SH = f"{OUT}/shard{SHARD}"
os.makedirs(SH, exist_ok=True)
print(f"-> chạy shard {SHARD}: {NQ} câu, {len(SHOPS)} shop")

In [ ]:
# smoke: 2 câu đầu của shop đầu shard. Thư mục tạm trỏ về dữ liệu public, chỉ thay questions.jsonl
if RUN_SMOKE:
    SM = "/tmp/smoke_pub"
    os.makedirs(SM, exist_ok=True)
    for name in os.listdir(PUB):
        if name != "questions.jsonl" and not os.path.exists(f"{SM}/{name}"):
            os.symlink(f"{PUB}/{name}", f"{SM}/{name}")
    rows = [l for l in open(f"{PUB}/questions.jsonl", encoding="utf8")
            if l.strip() and json.loads(l)["business_id"] == SHOPS[0]][:2]
    open(f"{SM}/questions.jsonl", "w", encoding="utf8").writelines(rows)
    assert sh(f"{RUN} --data {SM} --out /tmp/smoke_out/submission.jsonl") == 0, "smoke lỗi: gửi output cell này cho lead"
    print([json.loads(l)["type"] for l in open("/tmp/smoke_out/submission.jsonl", encoding="utf8")])

In [ ]:
# chạy chính: datamate.run trên các shop của shard; chạy nền, báo tiến độ mỗi phút, zip định kỳ
# (tiến độ nhảy theo shop: câu của một shop được gửi model cùng một lượt)
import shutil
import time
def pack(final=False):
    sh(f'uv pip freeze --python {PY} | grep -iE "^(vllm|torch|transformers|xgrammar)==" > {OUT}/versions.txt')
    sh(f"nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader >> {OUT}/versions.txt")
    sh(f"git rev-parse --short HEAD >> {OUT}/versions.txt")
    z = shutil.make_archive(f"/kaggle/working/datamate_public_{MODEL_NAME}{'_think' if THINKING else ''}_{TAG}"
                            f"_shard{SHARD}of{NSHARDS}", "zip", OUT)
    print(("ZIP " if final else "zip dở dang ") + z, os.path.getsize(z) // 1024, "KB", flush=True)

def done():
    try:
        return sum(1 for _ in open(f"{SH}/log.jsonl", encoding="utf8"))
    except FileNotFoundError:
        return 0

cmd = f"{RUN} --data {PUB} --out {SH}/submission.jsonl --log {SH}/log.jsonl --only {' '.join(SHOPS)}"
print("$", cmd)
t0 = last_zip = time.time()
with open(f"{SH}/run_public.txt", "w") as logf:
    p = subprocess.Popen(cmd, shell=True, stdout=logf, stderr=subprocess.STDOUT)
    while p.poll() is None:
        time.sleep(60)
        el, n = time.time() - t0, done()
        eta = f", ước còn {(NQ - n) * el / n / 3600:.1f} giờ" if n else ""
        print(f"{el / 60:6.0f} phút: {n}/{NQ} câu{eta}", flush=True)
        if time.time() - last_zip > ZIP_EVERY_MIN * 60:
            pack()
            last_zip = time.time()
print("exit", p.returncode, f"{(time.time() - t0) / 3600:.2f} giờ, {done()}/{NQ} câu")
sh(f"grep -E 'questions in|agent_calls|Error' {SH}/run_public.txt | tail -5")
assert p.returncode == 0, "datamate.run lỗi: gửi run_public.txt cho lead"

In [ ]:
# thống kê (chỉ số đếm, không in câu hỏi/đáp án) rồi đóng gói lần cuối: tải file ZIP này về
rows = [json.loads(l) for l in open(f"{SH}/submission.jsonl", encoding="utf8")]
src = Counter((json.loads(l).get("inputs") or {}).get("source") for l in open(f"{SH}/log.jsonl", encoding="utf8"))
print(len(rows), "dòng;", Counter(r["type"] for r in rows).most_common())
print("source:", src.most_common())
pack(final=True)